# Track A & B — key results and how they were obtained

Two correctness campaigns (2026-07-01), each landed as reviewed commits and
pinned by regression tests. Every result below is **recomputed live** in this
notebook.

| Track | What | Commits | Headline result |
|---|---|---|---|
| A | Indexer correctness | `607e59c`…`b01d00f` | Physical concentrations; forward/inverse share one emissivity; outer search minimises the inner objective; falsifiable T tests |
| B | Peak finder repair | `868a834`, `29f5960` | Full-path areas exact to ≤0.2% on ground truth (was +318% fast / +50% wings); noise-referenced detection that survives the corpus grid |
| A+B | End-to-end round trip | — | Composition recovered to ~0.3% relative at the true plasma state (was +20% on Al) |

**Method used throughout:** measure against synthetic ground truth first, fix,
re-measure, pin with a test, then adversarially review the diff with independent
verifier agents (their confirmed findings produced `3932367` and `29f5960`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import voigt_profile

from alibz import PeakyFinder, PeakyIndexer, PeakyMaker
from alibz.peaky_indexer_v3 import LineTable
from alibz.utils.sahaboltzmann import SahaBoltzmann

plt.rcParams.update({'font.size': 11})
DB = '../db'

## A.1 One emissivity convention for forward and inverse models

**Broken:** the indexer's per-line weights omitted the $hc/\lambda$
photon-energy factor that the forward synthesis applies — a smooth
$\sim$5$\times$ relative bias between blue and red lines *of the same
species* across 180–961 nm, which directly corrupts the temperature
inferred from relative line intensities.

**Fix:** both directions now call one shared function
(`alibz.utils.sahaboltzmann.line_emissivity`),

$$\varepsilon_j = \frac{1}{4\pi}\,\frac{hc}{\lambda_j}\, gA_j\,
e^{-E_{k,j}/k_BT}\, /\,Z_s(T)\; \cdot f_{\mathrm{stage},s}(T, n_e),$$

pinned line-for-line by `TestLineWeightConventionConsistency`, including a
literal-constants recomputation that fails if the shared function itself
regresses (mutation-verified).

In [ ]:
T_demo, logne_demo = 11_000.0, 17.0
sb = SahaBoltzmann(DB)
table = LineTable(sb.db, sb, wl_range=(390.0, 800.0), max_ion_stage=2, min_gA=100.0)
idx_demo = PeakyIndexer(np.array([[1.0, 500.0, 0.05, 0.05]]))
idx_demo.line_table = table
weights = idx_demo._line_weights(T_demo, logne_demo)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
for sp in table.species:
    if sp.element == 'Ca':
        span = slice(sp.line_start, sp.line_end)
        ax1.semilogy(table.wavelengths[span], weights[span], 'o', ms=5,
                     label=f'indexer weights: Ca {"I" * sp.ion}')
_, cal_int = sb.calculate('Ca', np.array([T_demo]), logne_demo)
cal_wl, _ = sb.calculate('Ca', np.array([T_demo]), logne_demo)
ax1.semilogy(cal_wl[0], cal_int[0], 'k+', ms=8, mew=0.8, alpha=0.6,
             label='forward model (SahaBoltzmann.calculate)')
ax1.set_xlim(390, 800); ax1.set_ylim(bottom=1e-2)
ax1.set_xlabel('wavelength [nm]'); ax1.set_ylabel('line emissivity [arb]')
ax1.set_title('Inverse weights == forward intensities (Ca)', fontweight='bold')
ax1.legend(fontsize=8)

lam = np.linspace(180, 961, 200)
ax2.plot(lam, (lam[0] / lam), 'r-')
ax2.set_xlabel('wavelength [nm]'); ax2.set_ylabel('relative weight factor')
ax2.set_title('The previously-omitted $hc/\\lambda$ factor\n'
              f'({lam[-1]/lam[0]:.1f}x blue-to-red span)', fontweight='bold')
plt.tight_layout(); plt.show()

## A.2 Physical concentrations and per-element composition

**Broken:** `FitResult.concentrations` were column-max-normalised NNLS
coefficients — each species scaled by its own strongest line intensity, not
comparable across species (and `bench/` treated them as abundances).

**Fix:** the solve un-normalises (`c = c_norm / col_max`), so concentrations
are physical scale factors usable with `forward_model()`. Ion stages stay
**independent unknowns** (each stage independently estimates total element
density because the design columns carry the Saha stage fractions — also the
right robustness choice for phase-heterogeneous targets where the same
element ionises differently per host mineral). Per-element values come from
a tied non-negative re-solve that reduces exactly to the
$\lVert A_{:,s}\rVert^2$ precision-weighted average for disjoint stage
support and stays order-invariant under blends (the adversarial review
caught the naive average flipping 0.4 ↔ 1.2 on identical physics).
`stage_disagreement` is the LTE/phase-heterogeneity diagnostic.

## A.3 The outer search minimises what the inner solve minimises

Degenerate trials returned $\lVert y\rVert$ (an L2 norm) while real trials
returned a sum of squares — so a trial that deactivated every species looked
orders of magnitude *better* to `gp_minimize`. And the evidence ridge
penalties were minimised by the NNLS but invisible to the outer cost. Both
fixed; T-recovery tests are now falsifiable (search 6–16 kK, assert ±600 K,
measured ~±110 K).

**Known limitation (by design, Track C):** with independent stages, $n_e$
has no handle on the fitted amplitudes — it is currently a *prior*, and the
Stark-width coupling under construction is what will make it a measurement.

## B. Peak finder repair — measured against ground truth

Baselines measured before the fix (synthetic Voigt fixtures, exact truth):

| Regime | Before | After |
|---|---|---|
| Fast/corpus path, any line | **+318%** area (height/2 stored as area) | **+8%** uniform (documented partition bias) |
| Full path, weak line 0.7 nm from a 100× line | +50% (unmodelled wing pedestal) | ≤0.2% |
| Strong lines w/ sensitive detection | split into duplicates (−62…−87%) | exact |
| Detection floor | everything below the *peak-population mean* dropped | noise-referenced |

Fixes: physical seeds ($A = h/V(0)$, Olivero–Longbothum width partition),
modelled-wing subtraction outside every fit window (strongest-first), one
explicit (key, seed) list for window bookkeeping (killed a parameter-permutation
bug), width-aware shoulder exclusion, one-sided contamination guard, and a
multi-lag noise scale that survives the corpus grid's interpolation upsampling
(the review caught the lag-1 version deflating 2–10×).

In [ ]:
def synth(lines, w_lo=390.0, w_hi=510.0, inc=0.01, noise=0.0, seed=7):
    x = np.arange(w_lo, w_hi + inc, inc)
    y = np.zeros_like(x)
    for area, mu, sig, gam in lines:
        y += area * voigt_profile(x - mu, sig, gam)
    if noise > 0:
        y = y + np.random.default_rng(seed).normal(0.0, noise, size=y.size)
    return x, y

finder = PeakyFinder.__new__(PeakyFinder)

# Fixture: one 10^6 line + weak satellites riding its Lorentzian wings
WINGS = [(1e6, 450.0, 0.03, 0.02)] + [
    (a, 450.0 + off, 0.03, 0.02)
    for off, a in [(0.5, 2e3), (1.0, 2e3), (2.0, 2e3), (-0.7, 1e4), (-1.5, 1e4)]]
x, y = synth(WINGS)
fit = finder.fit_spectrum(x, y, subtract_background=False, plot=False, n_sigma=0)
pk = fit['sorted_parameter_array']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
m = (x > 447.2) & (x < 453.0)
ax1.semilogy(x[m], np.clip(y[m], 1e-1, None), 'k-', lw=1.2, label='data')
ax1.semilogy(x[m], np.clip(fit['profile'][m], 1e-1, None), 'r--', lw=1.2, label='total fit')
for i, row in enumerate(pk):
    comp = row[0] * voigt_profile(x[m] - row[1], max(row[2], 1e-6), max(row[3], 1e-6))
    ax1.fill_between(x[m], np.clip(comp, 1e-1, None), 1e-1, alpha=0.25)
ax1.set_xlabel('wavelength [nm]'); ax1.set_ylabel('intensity')
ax1.set_title('Satellites on a 500x line: individually resolved', fontweight='bold')
ax1.legend(fontsize=8)

errs, labels = [], []
for area, mu, sig, gam in WINGS:
    j = int(np.argmin(np.abs(pk[:, 1] - mu)))
    errs.append((pk[j, 0] - area) / area * 100)
    labels.append(f'{mu:.1f}')
ax2.bar(range(len(errs)), errs, color='#2ca02c')
ax2.axhline(0, color='k', lw=0.8)
ax2.set_xticks(range(len(errs)), labels)
ax2.set_xlabel('line center [nm]'); ax2.set_ylabel('area error [%]')
ax2.set_ylim(-2, 2)
ax2.set_title(f'Full-path area errors (max |err| = {max(abs(e) for e in errs):.2f}%)',
              fontweight='bold')
plt.tight_layout(); plt.show()

### B.2 Noise-referenced detection that survives the corpus grid

Significance is measured against a robust point-noise scale, not the peak
population. The scale is a **multi-lag** difference estimate: on linearly
interpolated spectra (the corpus resamples native ~0.033 nm data onto a
0.01 nm grid) a lag-1 estimate deflates by exactly the upsampling factor.

In [ ]:
rng = np.random.default_rng(3)
sigma_true = 20.0
x_nat = np.arange(200.0, 400.0, 0.0333)
y_nat = rng.normal(0.0, sigma_true, size=x_nat.size)
x_up = np.arange(200.0, 399.9, 0.01)
y_up = np.interp(x_up, x_nat, y_nat)

lag1 = 1.4826 * np.median(np.abs(np.diff(y_up))) / np.sqrt(2)
multi = finder._noise_scale(y_up)
print(f'true sigma {sigma_true:.1f} | lag-1 on corpus grid {lag1:.1f} '
      f'(x{sigma_true/lag1:.1f} deflated) | multi-lag {multi:.1f}')

# Detection on a noisy spectrum: real lines kept, noise maxima rejected
lines = [(a, m, 0.03, 0.02) for a, m in zip([2e3, 8e3, 4e4, 2e5], np.linspace(400, 500, 4))]
xn, yn = synth(lines, noise=5.0)
fitn = finder.fit_spectrum(xn, yn, subtract_background=False, plot=False,
                           n_sigma=0, skip_profile=True)
pkn = fitn['sorted_parameter_array']
fig, ax = plt.subplots(figsize=(13, 3.5))
ax.plot(xn, yn, 'k-', lw=0.5, alpha=0.8)
hts = pkn[:, 0] * voigt_profile(0.0, np.maximum(pkn[:, 2], 1e-6), np.maximum(pkn[:, 3], 1e-6))
ax.scatter(pkn[:, 1], hts, color='r', s=25, zorder=3,
           label=f'{pkn.shape[0]} peaks kept (4 true lines, ~12000 noise maxima)')
ax.set_xlabel('wavelength [nm]'); ax.set_ylabel('intensity')
ax.set_title('Detection at 3-sigma noise floor', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

## A+B end-to-end: composition round trip

Synthesise Ca 0.60 / Al 0.40 at (T, $n_e$) = (10 kK, $10^{17}\,$cm$^{-3}$)
over 392–397.5 nm, fit peaks, index with the plasma state pinned at truth.
Before the two tracks: Al came out **+20%** (its weak Al I lines ride the
Ca II H&K Lorentzian wings, and the finder handed that pedestal to the
indexer). Now both elements recover to a few tenths of a percent.

In [ ]:
maker = PeakyMaker(DB)
T_true, ne_true = 10_000.0, 17.0
truth = {'Ca': 0.6, 'Al': 0.4}
fracs = np.zeros(maker.max_z)
for el, f in truth.items():
    fracs[maker.db.elements.index(el)] = f
xs, ys, _ = maker.peak_maker(fracs, w_lo=392.0, w_hi=397.5, inc=0.01,
                             voigt_sig=0.03, voigt_gam=0.02,
                             temperature=T_true, ne=ne_true)
fit = finder.fit_spectrum(xs, ys, subtract_background=False, plot=False, n_sigma=0)
result = PeakyIndexer(fit['sorted_parameter_array'],
                      temperature_init=T_true, ne_init=ne_true).run(
    shift_tolerance=0.05, max_ion_stage=2,
    T_bounds=(T_true - 1, T_true + 1), ne_bounds=(ne_true - 1e-3, ne_true + 1e-3),
    sigma_bounds=(0.02, 0.05), gamma_bounds=(0.01, 0.04),
    n_calls=6, verbose=False)

print(f'r^2 = {result.r_squared:.6f}')
for s, sp in enumerate(result.species):
    if result.concentrations[s] > 0:
        print(f'  {sp.element} {"I" * sp.ion}: c = {result.concentrations[s]:.4f}')
print('element_fractions:', {k: round(v, 4) for k, v in result.element_fractions.items()})
print('stage_disagreement:', result.stage_disagreement)

fig, ax = plt.subplots(figsize=(5, 3.5))
els = list(truth)
w = 0.35
ax.bar(np.arange(2) - w/2, [truth[e] for e in els], w, label='truth', color='0.6')
ax.bar(np.arange(2) + w/2, [result.element_fractions.get(e, 0) for e in els], w,
       label='recovered', color='#d62728')
ax.set_xticks(range(2), els); ax.set_ylabel('element fraction')
ax.set_title('Composition round trip (pinned plasma state)', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

## Reproducing / extending

- **Tests** pin everything shown here: `tests/test_physics_integration.py`
  (convention consistency, round trips, falsifiable T), `tests/test_peaky_indexer.py`
  (physical units, aggregation, objective consistency), `tests/test_peaky_finder_accuracy.py`
  (ground-truth accuracy, noise machinery). Run `pytest tests/ -q` (~1 min).
- **Real-data check:** `notebooks/peaky_finder_peaks.ipynb` runs the repaired
  finder on `data/remote_samples` (peak locations + per-peak profiles; note the
  cleanly Stark-dominated H$\alpha$ at 656.3 nm — the Track C anchor).
- **Next (Track C):** per-line Stark-width coupling
  $\gamma_j = \gamma_{\mathrm{inst}} + w_{S,j}\cdot n_e/n_{e,\mathrm{ref}}$
  to turn $n_e$ from a prior into a measurement, followed by the
  phase-assemblage closure loop.